# 05 - Cohort 1 zero citation check

This notebook creates the first descriptive panel check for the
final panel organized by researcher, conference, and year.

Cohort 1 means all researchers who are on the PC in the same
conference year being plotted. The figure shows the distribution of
citation counts for those PC service observations, with the grey
percentage above each year showing the share of PC service observations
with zero citations.

## 1. Setup

In [1]:
import sys
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "project_setup.py").exists() and (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break
else:
    raise RuntimeError(
        "Could not find the repository root. Launch Jupyter from the repo root "
        "or set PYTHONPATH to the folder containing project_setup.py."
    )

import os
import sys
from pathlib import Path

os.environ.setdefault("ARROW_USER_SIMD_LEVEL", "NONE")

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "config" / "project_config.yaml").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        break

import matplotlib as mpl
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib import font_manager

from project_setup import ensure_dirs, setup_project

setup = setup_project()
PROJECT = setup.project_folder

STEP_3_PREPARED = PROJECT / "step_3_data" / "prepared"
STEP_3_FIGURES = PROJECT / "step_3_artifacts" / "figures"
STEP_3_SUMMARY = PROJECT / "step_3_artifacts" / "summary_tables"
ensure_dirs(STEP_3_FIGURES, STEP_3_SUMMARY)

PANEL_PATH = STEP_3_PREPARED / "panel.parquet"
PANEL_FEATURES_PATH = STEP_3_PREPARED / "panel_features.parquet"
FIGURE_OUT = STEP_3_FIGURES / "A_sparkline_strip.pdf"
FIGURE_LOG_OUT = STEP_3_FIGURES / "A_sparkline_strip_log.pdf"
SUMMARY_OUT = STEP_3_SUMMARY / "A_sparkline_strip_summary.csv"
ZERO_CITED_ALL_OUT = STEP_3_SUMMARY / "zero_cited_pc_service_observations.csv"
ZERO_CITED_SUMMARY_OUT = STEP_3_SUMMARY / "zero_cited_pc_service_summary.csv"
ZERO_CITED_BY_CONFERENCE_OUT = STEP_3_SUMMARY / "zero_cited_by_conference.csv"
ZERO_CITED_BY_CONFIDENCE_OUT = STEP_3_SUMMARY / "zero_cited_by_match_confidence.csv"
ZERO_CITED_BY_BASIS_OUT = STEP_3_SUMMARY / "zero_cited_by_citation_basis.csv"
ZERO_CITED_BY_CONFERENCE_YEAR_OUT = STEP_3_SUMMARY / "zero_cited_by_conference_year.csv"

print(f"Project folder: {PROJECT}")
print(f"Panel input: {PANEL_PATH.relative_to(PROJECT)}")

Project folder: /Users/endersari/2026-02-citations-vs-pc-memberships
Panel input: step_3_data/prepared/panel.parquet


## 2. Plot style

In [2]:
font_path = PROJECT / "fonts" / "LinLibertine_R.ttf"
if font_path.exists():
    font_manager.fontManager.addfont(str(font_path))
    font_prop = font_manager.FontProperties(fname=font_path)
    font_family = font_prop.get_name()
else:
    font_family = "serif"

mpl.rcParams.update(
    {
        "axes.titlesize": 14,
        "axes.labelsize": 12,
        "font.size": 12,
        "legend.fontsize": 12,
        "xtick.labelsize": 12,
        "ytick.labelsize": 12,
        "font.family": font_family,
        "text.usetex": True,
    }
)

CONFERENCE_COLORS = {
    "ICFP": "#f4b8e4",
    "POPL": "#81c8be",
    "OOPSLA": "#7287fd",
    "OOPSLA1": "#9aa6f7",
    "OOPSLA2": "#5a72e0",
    "PLDI": "#e5c890",
}

## 3. Read the panel

In [3]:
panel = pd.read_parquet(PANEL_PATH)
panel_features = pd.read_parquet(PANEL_FEATURES_PATH)
panel["year"] = panel["year"].astype(int)
panel["pc_status"] = panel["pc_status"].astype(int)
panel["citation_count"] = panel["citation_count"].astype(float)

print(f"panel: {panel.shape}")
print(f"panel_features: {panel_features.shape}")
display(panel.head())

panel: (37128, 7)
panel_features: (37128, 39)


,panel_row_id,researcher_id,name,conference,year,pc_status,citation_count
0,aaronbembenek|ICFP|2017,aaronbembenek,Aaron Bembenek,ICFP,2017,0,0.0
1,aaronbembenek|ICFP|2018,aaronbembenek,Aaron Bembenek,ICFP,2018,0,0.0
2,aaronbembenek|ICFP|2019,aaronbembenek,Aaron Bembenek,ICFP,2019,0,0.0
3,aaronbembenek|ICFP|2020,aaronbembenek,Aaron Bembenek,ICFP,2020,0,0.0
4,aaronbembenek|ICFP|2021,aaronbembenek,Aaron Bembenek,ICFP,2021,0,0.0


## 4. Keep Cohort 1 observations

Cohort 1 is a PC year check. For each conference year, I keep the rows
where the researcher is on the PC for that same conference year. I also keep
only rows with available citation outcomes, because unresolved identity
cases do not have author level citation counts.

OOPSLA is shown as three panels: the single issue OOPSLA years before
2022, and the separate OOPSLA1 and OOPSLA2 rounds from 2022 onward.

In [4]:
STRANDS = [
    ("ICFP", ["ICFP"], CONFERENCE_COLORS["ICFP"]),
    ("POPL", ["POPL"], CONFERENCE_COLORS["POPL"]),
    ("OOPSLA", ["OOPSLA"], CONFERENCE_COLORS["OOPSLA"]),
    ("OOPSLA1", ["OOPSLA1"], CONFERENCE_COLORS["OOPSLA1"]),
    ("OOPSLA2", ["OOPSLA2"], CONFERENCE_COLORS["OOPSLA2"]),
    ("PLDI", ["PLDI"], CONFERENCE_COLORS["PLDI"]),
]

cohort1_all = panel.loc[panel["pc_status"].eq(1)].copy()
cohort1 = cohort1_all.loc[cohort1_all["citation_count"].notna()].copy()

rows = []
for label, confs, color in STRANDS:
    sub_all = cohort1_all.loc[cohort1_all["conference"].isin(confs)].copy()
    sub_plot = cohort1.loc[cohort1["conference"].isin(confs)].copy()
    for year, group_all in sub_all.groupby("year"):
        group_plot = sub_plot.loc[sub_plot["year"].eq(year)]
        rows.append(
            {
                "panel": label,
                "year": int(year),
                "n_pc_observations": int(len(group_all)),
                "n_plotted_observations": int(len(group_plot)),
                "n_missing_citation_count": int(group_all["citation_count"].isna().sum()),
                "n_pc_researchers": int(group_all["researcher_id"].nunique()),
                "n_plotted_researchers": int(group_plot["researcher_id"].nunique()),
                "n_zero_cited": int(group_plot["citation_count"].eq(0).sum()),
                "mean_citations": float(group_plot["citation_count"].mean()),
                "median_citations": float(group_plot["citation_count"].median()),
                "zero_share": float(group_plot["citation_count"].eq(0).mean()),
            }
        )

summary = pd.DataFrame(rows).sort_values(["panel", "year"])
combined_all = cohort1_all.loc[
    cohort1_all["conference"].isin(
        ["ICFP", "POPL", "OOPSLA", "OOPSLA1", "OOPSLA2", "PLDI"]
    )
]
combined_plot = cohort1.loc[
    cohort1["conference"].isin(
        ["ICFP", "POPL", "OOPSLA", "OOPSLA1", "OOPSLA2", "PLDI"]
    )
]
combined_summary = (
    combined_all
    .groupby("year")
    .agg(
        panel=("conference", lambda s: "All"),
        n_pc_observations=("panel_row_id", "size"),
        n_missing_citation_count=("citation_count", lambda s: int(s.isna().sum())),
        n_pc_researchers=("researcher_id", "nunique"),
    )
    .reset_index()
)
combined_plot_summary = (
    combined_plot
    .groupby("year")
    .agg(
        n_plotted_observations=("panel_row_id", "size"),
        n_plotted_researchers=("researcher_id", "nunique"),
        n_zero_cited=("citation_count", lambda s: int(s.eq(0).sum())),
        mean_citations=("citation_count", "mean"),
        median_citations=("citation_count", "median"),
        zero_share=("citation_count", lambda s: float(s.eq(0).mean())),
    )
    .reset_index()
)
combined_summary = combined_summary.merge(
    combined_plot_summary, on="year", how="left"
)
summary_out = pd.concat([combined_summary, summary], ignore_index=True)
summary_out.to_csv(SUMMARY_OUT, index=False)

full_panel = panel.merge(
    panel_features[
        [
            "panel_row_id",
            "citation_match_basis",
            "match_confidence",
            "panel_identity_status",
        ]
    ],
    on="panel_row_id",
    how="left",
)
full_cohort1_all = full_panel.loc[full_panel["pc_status"].eq(1)].copy()
full_cohort1 = full_cohort1_all.loc[
    full_cohort1_all["citation_count"].notna()
].copy()

zero_cited_all = (
    full_panel.loc[
        full_panel["pc_status"].eq(1)
        & full_panel["citation_count"].eq(0),
        [
            "panel_row_id",
            "conference",
            "year",
            "name",
            "researcher_id",
            "citation_count",
            "citation_match_basis",
            "match_confidence",
            "panel_identity_status",
        ],
    ]
    .sort_values(["conference", "year", "name"])
    .reset_index(drop=True)
)
zero_cited_all.to_csv(ZERO_CITED_ALL_OUT, index=False)

zero_cited_summary = (
    cohort1_all
    .groupby(["conference", "year"])
    .agg(
        n_pc_service_observations=("panel_row_id", "size"),
        n_missing_citation_count=("citation_count", lambda s: int(s.isna().sum())),
    )
    .reset_index()
    .merge(
        cohort1
        .groupby(["conference", "year"])
        .agg(
            n_citation_observations=("panel_row_id", "size"),
            n_zero_cited=("citation_count", lambda s: int(s.eq(0).sum())),
            zero_cited_share=("citation_count", lambda s: float(s.eq(0).mean())),
        )
        .reset_index(),
        on=["conference", "year"],
        how="left",
    )
    .sort_values(["conference", "year"])
    .reset_index(drop=True)
)
zero_cited_summary.to_csv(ZERO_CITED_SUMMARY_OUT, index=False)

zero_cited_by_conference = (
    cohort1_all
    .groupby("conference")
    .agg(
        pc_service_rows=("panel_row_id", "size"),
        missing_citation=("citation_count", lambda s: int(s.isna().sum())),
        pc_researchers=("researcher_id", "nunique"),
    )
    .reset_index()
    .merge(
        cohort1
        .groupby("conference")
        .agg(
            citation_rows=("panel_row_id", "size"),
            zero_rows=("citation_count", lambda s: int(s.eq(0).sum())),
            mean_citations=("citation_count", "mean"),
            median_citations=("citation_count", "median"),
            p75_citations=("citation_count", lambda s: float(s.quantile(0.75))),
            p90_citations=("citation_count", lambda s: float(s.quantile(0.90))),
            max_citations=("citation_count", "max"),
        )
        .reset_index(),
        on="conference",
        how="left",
    )
    .merge(
        zero_cited_all
        .groupby("conference")["researcher_id"]
        .nunique()
        .reset_index(name="zero_researchers"),
        on="conference",
        how="left",
    )
    .fillna({"zero_researchers": 0})
)
zero_cited_by_conference["zero_share"] = (
    zero_cited_by_conference["zero_rows"]
    / zero_cited_by_conference["citation_rows"]
)
zero_cited_by_conference = zero_cited_by_conference[
    [
        "conference",
        "pc_service_rows",
        "citation_rows",
        "missing_citation",
        "pc_researchers",
        "zero_rows",
        "zero_researchers",
        "zero_share",
        "mean_citations",
        "median_citations",
        "p75_citations",
        "p90_citations",
        "max_citations",
    ]
].sort_values("conference")
zero_cited_by_conference.to_csv(ZERO_CITED_BY_CONFERENCE_OUT, index=False)

zero_cited_by_confidence = (
    full_cohort1
    .groupby("match_confidence", dropna=False)
    .agg(
        citation_rows=("panel_row_id", "size"),
        zero_rows=("citation_count", lambda s: int(s.eq(0).sum())),
        pc_researchers=("researcher_id", "nunique"),
        mean_citations=("citation_count", "mean"),
        median_citations=("citation_count", "median"),
    )
    .reset_index()
    .merge(
        zero_cited_all
        .groupby("match_confidence", dropna=False)["researcher_id"]
        .nunique()
        .reset_index(name="zero_researchers"),
        on="match_confidence",
        how="left",
    )
    .fillna({"zero_researchers": 0})
)
zero_cited_by_confidence["zero_share"] = (
    zero_cited_by_confidence["zero_rows"]
    / zero_cited_by_confidence["citation_rows"]
)
zero_cited_by_confidence.to_csv(ZERO_CITED_BY_CONFIDENCE_OUT, index=False)

zero_cited_by_basis = (
    full_cohort1
    .groupby("citation_match_basis", dropna=False)
    .agg(
        citation_rows=("panel_row_id", "size"),
        zero_rows=("citation_count", lambda s: int(s.eq(0).sum())),
        pc_researchers=("researcher_id", "nunique"),
        mean_citations=("citation_count", "mean"),
        median_citations=("citation_count", "median"),
    )
    .reset_index()
    .merge(
        zero_cited_all
        .groupby("citation_match_basis", dropna=False)["researcher_id"]
        .nunique()
        .reset_index(name="zero_researchers"),
        on="citation_match_basis",
        how="left",
    )
    .fillna({"zero_researchers": 0})
)
zero_cited_by_basis["zero_share"] = (
    zero_cited_by_basis["zero_rows"]
    / zero_cited_by_basis["citation_rows"]
)
zero_cited_by_basis.to_csv(ZERO_CITED_BY_BASIS_OUT, index=False)

zero_cited_by_conference_year = (
    cohort1
    .groupby(["conference", "year"])
    .agg(
        citation_rows=("panel_row_id", "size"),
        zero_rows=("citation_count", lambda s: int(s.eq(0).sum())),
        mean_citations=("citation_count", "mean"),
        median_citations=("citation_count", "median"),
    )
    .reset_index()
)
zero_cited_by_conference_year["zero_share"] = (
    zero_cited_by_conference_year["zero_rows"]
    / zero_cited_by_conference_year["citation_rows"]
)
zero_cited_by_conference_year.to_csv(
    ZERO_CITED_BY_CONFERENCE_YEAR_OUT, index=False
)

print(f"Cohort 1 PC-service observations: {len(cohort1_all):,}")
print(f"Cohort 1 plotted observations:    {len(cohort1):,}")
print(f"Wrote {SUMMARY_OUT.relative_to(PROJECT)}")
print(f"Wrote {ZERO_CITED_ALL_OUT.relative_to(PROJECT)}")
print(f"Wrote {ZERO_CITED_SUMMARY_OUT.relative_to(PROJECT)}")
print(f"Wrote {ZERO_CITED_BY_CONFERENCE_OUT.relative_to(PROJECT)}")
print(f"Wrote {ZERO_CITED_BY_CONFIDENCE_OUT.relative_to(PROJECT)}")
print(f"Wrote {ZERO_CITED_BY_BASIS_OUT.relative_to(PROJECT)}")
print(f"Wrote {ZERO_CITED_BY_CONFERENCE_YEAR_OUT.relative_to(PROJECT)}")
display(summary_out.head(12))
display(zero_cited_by_conference)
display(zero_cited_by_basis)
display(zero_cited_all.head(12))

Cohort 1 PC-service observations: 2,499
Cohort 1 plotted observations:    2,485
Wrote step_3_artifacts/summary_tables/A_sparkline_strip_summary.csv
Wrote step_3_artifacts/summary_tables/zero_cited_pc_service_observations.csv
Wrote step_3_artifacts/summary_tables/zero_cited_pc_service_summary.csv
Wrote step_3_artifacts/summary_tables/zero_cited_by_conference.csv
Wrote step_3_artifacts/summary_tables/zero_cited_by_match_confidence.csv
Wrote step_3_artifacts/summary_tables/zero_cited_by_citation_basis.csv
Wrote step_3_artifacts/summary_tables/zero_cited_by_conference_year.csv


,year,panel,n_pc_observations,n_missing_citation_count,n_pc_researchers,n_plotted_observations,n_plotted_researchers,n_zero_cited,mean_citations,median_citations,zero_share
0,2017,All,90,2,87,88,85,22,5.090909,3.0,0.250000
1,2018,All,138,0,126,138,126,21,7.731884,5.0,0.152174
2,2019,All,143,1,134,142,133,9,6.211268,4.0,0.063380
3,2020,All,146,0,137,146,137,19,6.541096,5.0,0.130137
4,2021,All,259,0,237,259,237,38,6.463320,4.0,0.146718
5,2022,All,311,0,230,311,230,67,4.382637,3.0,0.215434
6,2023,All,407,1,289,406,288,104,4.477833,3.0,0.256158
7,2024,All,493,0,337,493,337,98,5.099391,3.0,0.198783
8,2025,All,512,10,361,502,355,117,4.498008,2.0,0.233068
9,2017,ICFP,23,0,23,23,23,4,6.608696,3.0,0.173913


,conference,pc_service_rows,citation_rows,missing_citation,pc_researchers,zero_rows,zero_researchers,zero_share,mean_citations,median_citations,p75_citations,p90_citations,max_citations
0,ICFP,318,317,1,250,67,60,0.211356,4.334385,3.0,6.0,11.0,30.0
1,OOPSLA,176,174,2,133,24,23,0.137931,6.132184,4.0,8.0,14.0,37.0
2,OOPSLA1,348,344,4,298,139,130,0.404070,2.238372,1.0,3.0,6.0,33.0
3,OOPSLA2,348,344,4,298,49,48,0.142442,5.651163,3.0,8.0,13.0,67.0
4,PLDI,786,783,3,480,154,140,0.196679,5.197957,3.0,7.0,13.0,45.0
5,POPL,523,523,0,378,62,55,0.118547,7.177820,4.0,10.0,17.0,90.0


,citation_match_basis,citation_rows,zero_rows,pc_researchers,mean_citations,median_citations,zero_researchers,zero_share
0,ambiguous_name_match,226,40,84,5.535398,4.0,30,0.176991
1,manual_accepted_name,122,29,51,3.975410,3.0,16,0.237705
2,name_only_match,25,14,13,0.800000,0.0,9,0.560000
3,openalex_author_id,2112,412,791,5.313920,3.0,309,0.195076


,panel_row_id,conference,year,name,researcher_id,citation_count,citation_match_basis,match_confidence,panel_identity_status
0,alexandrasilva|ICFP|2017,ICFP,2017,Alexandra Silva,alexandrasilva,0.0,ambiguous_name_match,RED,ambiguous_openalex_ids
1,kathrynegray|ICFP|2017,ICFP,2017,Kathryn E. Gray,kathrynegray,0.0,openalex_author_id,YELLOW,automatic_openalex_id
2,lindseykuper|ICFP|2017,ICFP,2017,Lindsey Kuper,lindseykuper,0.0,openalex_author_id,GREEN,automatic_openalex_id
3,martinerwig|ICFP|2017,ICFP,2017,Martin Erwig,martinerwig,0.0,ambiguous_name_match,RED,ambiguous_openalex_ids
4,alejandrorusso|ICFP|2018,ICFP,2018,Alejandro Russo,alejandrorusso,0.0,openalex_author_id,YELLOW,automatic_openalex_id
5,katsuhiroueno|ICFP|2018,ICFP,2018,Katsuhiro Ueno,katsuhiroueno,0.0,openalex_author_id,YELLOW,automatic_openalex_id
6,marcogaboardi|ICFP|2018,ICFP,2018,Marco Gaboardi,marcogaboardi,0.0,openalex_author_id,YELLOW,automatic_openalex_id
7,sandrineblazy|ICFP|2018,ICFP,2018,Sandrine Blazy,sandrineblazy,0.0,openalex_author_id,YELLOW,automatic_openalex_id
8,jenniferpaykin|ICFP|2019,ICFP,2019,Jennifer Paykin,jenniferpaykin,0.0,openalex_author_id,YELLOW,automatic_openalex_id
9,lauramcastro|ICFP|2019,ICFP,2019,Laura M. Castro,lauramcastro,0.0,openalex_author_id,YELLOW,automatic_openalex_id


## 5. Render the figure

In [5]:
cohort1["citation_count_log"] = np.log10(cohort1["citation_count"] + 1)


def shifted_geometric_mean(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    if len(values) == 0:
        return np.nan
    return float(10 ** np.mean(np.log10(values + 1)) - 1)


def panel_data(label: str, confs: list[str], y_column: str):
    sub_all = cohort1_all.loc[cohort1_all["conference"].isin(confs)].copy()
    sub_plot = cohort1.loc[cohort1["conference"].isin(confs)].copy()
    years = sorted(sub_all["year"].unique())
    by_year = [
        sub_plot.loc[sub_plot["year"].eq(year), y_column].to_numpy()
        for year in years
    ]
    means = [values.mean() if len(values) else np.nan for values in by_year]
    zero_pcts = [
        float(
            sub_plot.loc[sub_plot["year"].eq(year), "citation_count"]
            .eq(0)
            .mean()
            * 100
        )
        if len(sub_plot.loc[sub_plot["year"].eq(year)]) else 0.0
        for year in years
    ]
    return sub_all, sub_plot, years, by_year, means, zero_pcts


def add_jitter(ax, years, by_year, color, width=0.18, size=15, alpha=0.60):
    rng = np.random.default_rng(0)
    for year, values in zip(years, by_year):
        if len(values) == 0:
            continue
        xs = year + rng.uniform(-width, width, size=len(values))
        ax.scatter(
            xs,
            values,
            s=size,
            color=color,
            alpha=alpha,
            edgecolors="none",
            zorder=2.5,
        )


def draw_box_panel(ax, years, by_year, color):
    ax.boxplot(
        by_year,
        positions=years,
        widths=0.65,
        patch_artist=True,
        showfliers=False,
        medianprops=dict(color="black", linewidth=1.4),
        whiskerprops=dict(color="#444444", linewidth=1.0),
        capprops=dict(color="#444444", linewidth=1.0),
        boxprops=dict(
            facecolor=color,
            edgecolor="#333333",
            linewidth=0.8,
            alpha=0.55,
        ),
    )
    add_jitter(ax, years, by_year, color)


def annotate_zero_share(ax, years, by_year, zero_pcts, fontsize=7):
    if not years:
        return
    ymax = ax.get_ylim()[1]
    ax.set_ylim(top=ymax * 1.10)
    y_text = ymax * 1.02
    for year, values, zero_pct in zip(years, by_year, zero_pcts):
        if len(values) == 0:
            continue
        ax.annotate(
            f"{zero_pct:.0f}\%",
            xy=(year, y_text),
            ha="center",
            va="bottom",
            fontsize=fontsize,
            color="#666666",
        )


def style_axis(
    ax,
    years,
    title,
    sub_all,
    sub_plot,
    include_mean_title=True,
    mean_title="arithmetic",
    ylabel=True,
    y_label="Citation Count",
):
    n_obs = len(sub_plot)
    n_missing = int(sub_all["citation_count"].isna().sum())
    n_researchers = sub_all["researcher_id"].nunique()
    zero_pct = sub_plot["citation_count"].eq(0).mean() * 100 if n_obs else 0
    if include_mean_title:
        if mean_title == "shifted_geometric":
            mean_value = shifted_geometric_mean(sub_plot["citation_count"])
            mean_label = "geom. mean citations"
        else:
            mean_value = sub_plot["citation_count"].mean() if n_obs else 0
            mean_label = "mean citations"
        title_text = (
            f"{title} (N={n_researchers}, {mean_label}={mean_value:.1f}, "
            f"zero-cited={zero_pct:.0f}\%)"
        )
    else:
        title_text = f"{title} (N={n_researchers}, zero-cited={zero_pct:.0f}\%)"
    ax.set_title(title_text, fontsize=8.3, loc="left")
    ax.set_xlabel("Year", fontsize=9)
    if ylabel:
        ax.set_ylabel(y_label, fontsize=8.5, labelpad=2)
    ax.grid(color="#DDDDDD", linestyle=":", linewidth=0.6, axis="y")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if years:
        ax.set_xlim(min(years) - 0.6, max(years) + 0.6)
        ax.set_xticks(years)
        ax.tick_params(axis="x", rotation=0, labelsize=8)
        ax.tick_params(axis="y", labelsize=8)


def render_figure(
    figure_out: Path,
    y_column: str,
    y_label: str,
    include_mean_title: bool,
    mean_title: str,
    show_mean: bool,
):
    fig = plt.figure(figsize=(16, 9.4))
    grid = gridspec.GridSpec(
        3,
        3,
        figure=fig,
        height_ratios=[1.18, 2.25, 2.25],
        hspace=0.50,
        wspace=0.30,
    )

    ax_top = fig.add_subplot(grid[0, :])
    combined_all = pd.concat(
        [
            cohort1_all.loc[cohort1_all["conference"].isin(confs)].copy()
            for _, confs, _ in STRANDS
        ],
        ignore_index=True,
    )
    combined = pd.concat(
        [
            cohort1.loc[cohort1["conference"].isin(confs)].copy()
            for _, confs, _ in STRANDS
        ],
        ignore_index=True,
    )
    years_all = sorted(combined["year"].unique())
    by_year_all = [
        combined.loc[combined["year"].eq(year), y_column].to_numpy()
        for year in years_all
    ]
    means_all = [values.mean() if len(values) else np.nan for values in by_year_all]
    zero_all = [
        float(
            combined.loc[combined["year"].eq(year), "citation_count"]
            .eq(0)
            .mean()
            * 100
        )
        if len(combined.loc[combined["year"].eq(year)]) else 0.0
        for year in years_all
    ]

    if years_all:
        ax_top.boxplot(
            by_year_all,
            positions=years_all,
            widths=0.65,
            patch_artist=True,
            showfliers=False,
            medianprops=dict(color="black", linewidth=1.4),
            whiskerprops=dict(color="#444444", linewidth=1.0),
            capprops=dict(color="#444444", linewidth=1.0),
            boxprops=dict(
                facecolor="#BBBBBB",
                edgecolor="#333333",
                linewidth=0.8,
                alpha=0.60,
            ),
        )
        add_jitter(ax_top, years_all, by_year_all, "#888888")
        if show_mean:
            ax_top.scatter(
                years_all,
                means_all,
                marker="D",
                color="black",
                s=22,
                zorder=4,
                edgecolors="white",
                linewidths=0.8,
            )
        annotate_zero_share(ax_top, years_all, by_year_all, zero_all, fontsize=7)

    overall_zero = combined["citation_count"].eq(0).mean() * 100 if len(combined) else 0
    if include_mean_title:
        if mean_title == "shifted_geometric":
            overall_mean = shifted_geometric_mean(combined["citation_count"])
            mean_label = "geom. mean citations"
        else:
            overall_mean = combined["citation_count"].mean() if len(combined) else 0
            mean_label = "mean citations"
        top_title = (
            f"All conferences combined (obs={len(combined)}, "
            f"{mean_label}={overall_mean:.1f}, zero-cited={overall_zero:.0f}\%)"
        )
    else:
        top_title = (
            f"All conferences combined (obs={len(combined)}, "
            f"zero-cited={overall_zero:.0f}\%)"
        )
    ax_top.set_title(top_title, fontsize=9.5, loc="left")
    ax_top.set_xlabel("Year", fontsize=9)
    ax_top.set_ylabel(y_label, fontsize=8.5, labelpad=2)
    ax_top.grid(color="#DDDDDD", linestyle=":", linewidth=0.6, axis="y")
    ax_top.spines["top"].set_visible(False)
    ax_top.spines["right"].set_visible(False)
    if years_all:
        ax_top.set_xlim(min(years_all) - 0.6, max(years_all) + 0.6)
        ax_top.set_xticks(years_all)
        ax_top.tick_params(axis="x", labelsize=8)
        ax_top.tick_params(axis="y", labelsize=8)

    bottom_axes = [
        fig.add_subplot(grid[1, 0]),
        fig.add_subplot(grid[1, 1]),
        fig.add_subplot(grid[1, 2]),
        fig.add_subplot(grid[2, 0]),
        fig.add_subplot(grid[2, 1]),
        fig.add_subplot(grid[2, 2]),
    ]
    for ax, (label, confs, color) in zip(bottom_axes, STRANDS):
        sub_all, sub_plot, years, by_year, means, zero_pcts = panel_data(
            label, confs, y_column
        )
        if years:
            draw_box_panel(ax, years, by_year, color)
            if show_mean:
                ax.scatter(
                    years,
                    means,
                    marker="D",
                    color="black",
                    s=22,
                    zorder=4,
                    edgecolors="white",
                    linewidths=0.8,
                )
            annotate_zero_share(ax, years, by_year, zero_pcts, fontsize=6.5)
        style_axis(
            ax,
            years,
            label,
            sub_all,
            sub_plot,
            include_mean_title=include_mean_title,
            mean_title=mean_title,
            ylabel=True,
            y_label=y_label,
        )

    fig.savefig(figure_out, bbox_inches="tight", pad_inches=0.03)
    plt.close(fig)

    print(f"Wrote {figure_out.relative_to(PROJECT)}")


render_figure(
    FIGURE_OUT,
    y_column="citation_count",
    y_label="Citation Count",
    include_mean_title=True,
    mean_title="arithmetic",
    show_mean=True,
)
render_figure(
    FIGURE_LOG_OUT,
    y_column="citation_count_log",
    y_label="log10(citation count + 1)",
    include_mean_title=True,
    mean_title="shifted_geometric",
    show_mean=True,
)

Wrote step_3_artifacts/figures/A_sparkline_strip.pdf
Wrote step_3_artifacts/figures/A_sparkline_strip_log.pdf
